# PyTorch Neural Network Fundamentals — Class-Based Training

This notebook builds the same regression problem in two stages:

1. Manual linear regression — to understand weights, bias, loss, gradients, and gradient descent.
2. A PyTorch class-based neural network — to understand `nn.Module`, `forward()`, loss, backpropagation, and optimizer updates.

The dataset has **1 input feature**, so the model architecture must start with `nn.Linear(1, ...)`.


In [ ]:
# Install only if needed
# !pip install torchviz


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

# Optional: for visualizing the computation graph
# from torchviz import make_dot


## 1. Create the dataset

We create data approximately following:

$$y = 1 + 2x + noise$$

So the model should learn a weight close to **2** and a bias close to **1**.


In [ ]:
true_b = 1.0
true_w = 2.0
N = 100

np.random.seed(42)

X = np.random.rand(N, 1)
noise = 0.1 + np.random.randn(N, 1)
y = true_b + true_w * X + noise

print("X shape:", X.shape)
print("y shape:", y.shape)


In [ ]:
# Shuffle and split into training and validation sets

idx = np.arange(N)
np.random.shuffle(idx)

train_size = int(N * 0.8)

train_idx = idx[:train_size]
val_idx = idx[train_size:]

X_train, y_train = X[train_idx], y[train_idx]
X_val, y_val = X[val_idx], y[val_idx]

print("Training:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)


In [ ]:
plt.scatter(X_train, y_train, label="Train")
plt.scatter(X_val, y_val, label="Validation")
plt.xlabel("X")
plt.ylabel("y")
plt.legend()
plt.show()


# Part A — Understand the model manually

Before using PyTorch's automatic differentiation, understand the basic model:

$$\hat y = wx + b$$

where:

- `w` = weight
- `b` = bias
- `x` = input
- `y_hat` = prediction


In [ ]:
# Initialize parameters
w = 0.0
b = 0.0

# Forward pass
y_hat = w * X_train + b

# Mean Squared Error
error = y_hat - y_train
loss = np.mean(error ** 2)

print("Initial loss:", loss)


In [ ]:
# Manual gradients

b_grad = 2 * error.mean()
w_grad = 2 * (X_train * error).mean()

print("b gradient:", b_grad)
print("w gradient:", w_grad)


In [ ]:
# One manual gradient-descent update

lr = 0.1

w = w - lr * w_grad
b = b - lr * b_grad

print("Updated weight:", w)
print("Updated bias:", b)


# Part B — Convert NumPy arrays to PyTorch tensors

PyTorch works with tensors.

We also select CPU/GPU automatically.


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

X_train_tensor = torch.as_tensor(X_train, dtype=torch.float32).to(device)
y_train_tensor = torch.as_tensor(y_train, dtype=torch.float32).to(device)

X_val_tensor = torch.as_tensor(X_val, dtype=torch.float32).to(device)
y_val_tensor = torch.as_tensor(y_val, dtype=torch.float32).to(device)


# Part C — Build the neural network using a class

This is the most important pattern to learn:

```python
class MyModel(nn.Module):

    def __init__(self):
        super().__init__()
        # define layers here

    def forward(self, x):
        # define how data flows through layers
        return x
```

### Why `nn.Module`?

`nn.Module` is the base class for PyTorch models. It lets PyTorch automatically track trainable parameters and model layers.


In [ ]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()

        # 1 input feature -> 16 neurons
        self.layer1 = nn.Linear(1, 16)

        # Activation function
        self.relu = nn.ReLU()

        # 16 neurons -> 8 neurons
        self.layer2 = nn.Linear(16, 8)

        # 8 neurons -> 1 output
        self.output = nn.Linear(8, 1)

    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)

        x = self.layer2(x)
        x = self.relu(x)

        x = self.output(x)

        return x


model = MyModel().to(device)

print(model)


## How this network is formed

Your input has **1 feature**:

```text
x
↓
Linear(1 → 16)
↓
ReLU
↓
Linear(16 → 8)
↓
ReLU
↓
Linear(8 → 1)
↓
Prediction
```

Each `nn.Linear` layer contains trainable **weights and biases**.

The `forward()` method defines the path that the input follows through the network.


In [ ]:
# Inspect trainable parameters

for name, parameter in model.named_parameters():
    print(name, parameter.shape)


# Part D — Loss function and optimizer

For regression, Mean Squared Error is a common choice.

The optimizer changes the model's trainable parameters using their gradients.


In [ ]:
loss_fn = nn.MSELoss()

optimizer = optim.SGD(
    model.parameters(),
    lr=0.01
)

n_epochs = 1000


# Part E — Train the model

This is the core PyTorch training loop:

```text
1. Clear old gradients
2. Forward pass
3. Calculate loss
4. Backpropagation
5. Update parameters
6. Repeat
```


In [ ]:
train_losses = []
val_losses = []

for epoch in range(n_epochs):

    # Training mode
    model.train()

    # 1. Clear gradients from the previous iteration
    optimizer.zero_grad()

    # 2. Forward pass
    y_hat = model(X_train_tensor)

    # 3. Calculate loss
    loss = loss_fn(y_hat, y_train_tensor)

    # 4. Backpropagation
    loss.backward()

    # 5. Update weights and biases
    optimizer.step()

    # Validation
    model.eval()

    with torch.no_grad():
        val_pred = model(X_val_tensor)
        val_loss = loss_fn(val_pred, y_val_tensor)

    train_losses.append(loss.item())
    val_losses.append(val_loss.item())

    if (epoch + 1) % 100 == 0:
        print(
            f"Epoch {epoch + 1:4d} | "
            f"Train Loss: {loss.item():.4f} | "
            f"Val Loss: {val_loss.item():.4f}"
        )


# Part F — Evaluate the trained model

After training, we use `model.eval()` and `torch.no_grad()` because we only want predictions; we do not need gradients.


In [ ]:
model.eval()

with torch.no_grad():
    train_predictions = model(X_train_tensor)
    val_predictions = model(X_val_tensor)

print("Final train MSE:",
      loss_fn(train_predictions, y_train_tensor).item())

print("Final validation MSE:",
      loss_fn(val_predictions, y_val_tensor).item())


In [ ]:
# Plot training and validation loss

plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()


In [ ]:
# Plot predictions against the original data

X_all_tensor = torch.as_tensor(X, dtype=torch.float32).to(device)

model.eval()
with torch.no_grad():
    predictions = model(X_all_tensor).cpu().numpy()

plt.scatter(X, y, label="Actual")
plt.scatter(X, predictions, label="Prediction")
plt.xlabel("X")
plt.ylabel("y")
plt.legend()
plt.show()


# Part G — The most important functions to remember

| Function / class | Purpose |
|---|---|
| `nn.Module` | Base class for custom PyTorch models |
| `__init__()` | Define layers and model components |
| `forward()` | Define how input moves through the network |
| `nn.Linear()` | Fully connected layer |
| `nn.ReLU()` | Activation function |
| `model(x)` | Runs the forward pass |
| `nn.MSELoss()` | Mean Squared Error loss |
| `optimizer = optim.SGD(...)` | Creates the optimizer |
| `optimizer.zero_grad()` | Clears old gradients |
| `loss.backward()` | Calculates gradients using backpropagation |
| `optimizer.step()` | Updates weights and biases |
| `model.train()` | Training mode |
| `model.eval()` | Evaluation mode |
| `torch.no_grad()` | Disable gradient calculation during inference |
| `model.parameters()` | Gives trainable model parameters |
| `.to(device)` | Moves model/tensors to CPU or GPU |
| `.item()` | Converts a single-value tensor to a Python number |


# Final mental model

The complete process is:

```text
Input
  ↓
forward()
  ↓
Prediction
  ↓
Loss
  ↓
loss.backward()
  ↓
Gradients
  ↓
optimizer.step()
  ↓
Updated weights
  ↓
Repeat for many epochs
```

### Class-based PyTorch template

You should learn this template:

```python
class MyModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.layer1 = nn.Linear(input_features, hidden_units)
        self.relu = nn.ReLU()
        self.output = nn.Linear(hidden_units, output_features)

    def forward(self, x):

        x = self.layer1(x)
        x = self.relu(x)
        x = self.output(x)

        return x


model = MyModel()

loss_fn = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

for epoch in range(epochs):

    optimizer.zero_grad()

    prediction = model(X)

    loss = loss_fn(prediction, y)

    loss.backward()

    optimizer.step()
```

This template is the foundation you will reuse when you move from simple regression to larger PyTorch models and eventually CNNs.


### Using the Sequential Model : -> only defime how the layers are connected 

- -> Build The Model :

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim


# 1. Build the model using a class
class MyModel(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(1, 16),
            nn.ReLU(),

            nn.Linear(16, 8),
            nn.ReLU(),

            nn.Linear(8, 1)
        )

    def forward(self, x):
        return self.network(x)


# 2. Create the model
model = MyModel()

print(model)

: 

- -> Train the Model :

In [ ]:
# Loss function
loss_fn = nn.MSELoss()

# Optimizer
optimizer = optim.SGD(
    model.parameters(),
    lr=0.01
)

# Number of epochs
n_epochs = 1000


# Training loop
for epoch in range(n_epochs):

    # Training mode
    model.train()

    # Clear old gradients
    optimizer.zero_grad()

    # Forward pass
    y_pred = model(X_train_tensor)

    # Calculate loss
    loss = loss_fn(y_pred, y_train_tensor)

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    # Validation
    model.eval()

    with torch.no_grad():
        y_val_pred = model(X_val_tensor)
        val_loss = loss_fn(y_val_pred, y_val_tensor)

    if (epoch + 1) % 100 == 0:
        print(
            f"Epoch [{epoch+1}/{n_epochs}] "
            f"Train Loss: {loss.item():.4f} "
            f"Val Loss: {val_loss.item():.4f}"
        )